In [ ]:
import asyncio
import json
import time
import uuid
from collections.abc import Callable, Coroutine
from dataclasses import asdict, dataclass, field
from typing import Any, Literal, NamedTuple

from dotenv import load_dotenv
from genai_core import LLMService, ToolBudget
from google.genai import Client
from pydantic import BaseModel, Field, model_validator
from research_and_support_agent import BoundedAgent

load_dotenv()

In [ ]:
llm_client = Client()

### Task 1

implemented in **test_bounded_orchestrator.py**

### Task 2

In [ ]:
class SubTask(BaseModel):
    id: str
    objective: str
    output_format: str
    boundaries: str
    max_steps: int = Field(default=6, ge=1, le=15)


class WorkerResult(BaseModel):
    task_id: str
    status: str  # "ok" | "failed" | "timeout"
    summary: str = ""
    sources: list[str] = Field(default_factory=list)
    steps_used: int = 0
    tokens_used: int = 0


@dataclass
class Budget:
    max_tokens: int
    used: int = 0
    lock: asyncio.Lock = field(default_factory=asyncio.Lock)

    async def charge(self, n: int) -> bool:
        async with self.lock:
            if self.used + n > self.max_tokens:
                return False
            self.used += n
            return True


async def run_worker(
    task: SubTask,
    worker_fn: Callable[[SubTask], Coroutine[Any, Any, WorkerResult]],
    budget: Budget,
    timeout_s: float,
) -> WorkerResult:
    """Bounded worker wrapper: isolated execution, timeout, and budget enforcement."""
    try:
        result = await asyncio.wait_for(worker_fn(task), timeout=timeout_s)
    except TimeoutError:
        return WorkerResult(task_id=task.id, status="timeout")
    except Exception as exc:  # noqa
        return WorkerResult(task_id=task.id, status="failed", summary=str(exc)[:200])

    if not await budget.charge(result.tokens_used):
        return WorkerResult(task_id=task.id, status="failed", summary="budget exceeded")
    return result


async def orchestrate(
    subtasks: list[SubTask],
    worker_fn: Callable[[SubTask], Coroutine[Any, Any, WorkerResult]],
    budget: Budget,
    max_parallel: int = 3,
    timeout_s: float = 60.0,
) -> list[WorkerResult]:
    sem = asyncio.Semaphore(max_parallel)  # concurrency cap

    async def guarded(t: SubTask) -> WorkerResult:
        async with sem:
            return await run_worker(t, worker_fn, budget, timeout_s)

    return list(await asyncio.gather(*(guarded(t) for t in subtasks)))

In [20]:
# Module 17 Agent Worker using Dependency Injection
agent = BoundedAgent(llm_service=LLMService(), memory_manager=None)


async def module17_agent_worker(task: SubTask, agent: BoundedAgent = agent) -> WorkerResult:
    """Runs a Module 17 agent loop bounded by task.max_steps."""
    prompt = (
        f"Objective: {task.objective} | "
        f"Output format: {task.output_format} | "
        f"Boundaries: {task.boundaries}"
    )
    result = await agent.run(query=prompt, budget=ToolBudget(max_steps=task.max_steps))
    status_str = getattr(result.status, "value", str(result.status))
    return WorkerResult(
        task_id=task.id,
        status="ok" if status_str == "completed" else "failed",
        summary=result.final_text or "",
        steps_used=result.iterations,
        tokens_used=result.total_tokens,
    )

In [19]:
# Define three real subtasks to run in parallel
subtasks = [
    SubTask(
        id="subtask_1",
        objective="Compare multi-agent vs single-agent systems in 2 concise bullets.",
        output_format="2 bullet points",
        boundaries="Under 35 words",
        max_steps=3,
    ),
    SubTask(
        id="subtask_2",
        objective="Define the orchestrator-worker pattern in 2 concise bullets.",
        output_format="2 bullet points",
        boundaries="Under 35 words",
        max_steps=3,
    ),
    SubTask(
        id="subtask_3",
        objective="Explain why token budgets matter for agents in 2 concise bullets.",
        output_format="2 bullet points",
        boundaries="Under 35 words",
        max_steps=3,
    ),
]

budget = Budget(max_tokens=30000)

# Run all 3 subtasks in parallel
results = await orchestrate(
    subtasks=subtasks,
    worker_fn=module17_agent_worker,
    budget=budget,
    max_parallel=3,
    timeout_s=60.0,
)

for r in results:
    print(f"[{r.task_id}] status={r.status} | steps={r.steps_used} | tokens={r.tokens_used}")
    print(f"{r.summary}\n")

print(f"Total budget tokens consumed: {budget.used}")

[subtask_1] status=ok | steps=3 | tokens=5797
* **Single-agent systems:** Handle simpler, linear tasks efficiently with lower complexity and coordination overhead.
* **Multi-agent systems:** Excel at complex, modular workflows by distributing specialized tasks among collaborating agents.

[subtask_2] status=ok | steps=3 | tokens=5804
* A central orchestrator breaks down goals and delegates subtasks to specialist workers.
* The orchestrator coordinates workflows and combines results while workers execute specific steps.

[subtask_3] status=ok | steps=3 | tokens=5916
* **Controls API Costs:** Prevents autonomous agents from rapidly consuming excessive funds through endless tool calls and retries.
* **Maintains Performance:** Keeps context windows within optimal limits to avoid latency, degradation, and memory overflow.

Total budget tokens consumed: 17517


### Task 3


In [24]:
# --- Strictly Typed Contracts with Validation ---


class SubTask(BaseModel):
    """Input contract for delegating work with bounded steps."""

    id: str = Field(..., min_length=1, description="Unique task identifier")
    objective: str = Field(..., min_length=3, description="Precise objective")
    output_format: str = Field(..., description="Expected output schema")
    boundaries: str = Field(default="", description="Constraints or forbidden sources")
    max_steps: int = Field(default=6, ge=1, le=15, description="Bounded step limit [1, 15]")


class WorkerResult(BaseModel):
    """Output contract returned by workers to orchestrator."""

    task_id: str = Field(..., min_length=1)
    status: Literal["ok", "failed", "timeout"] = Field(..., description="Execution status")
    summary: str = Field(default="", description="Findings summary or error details")
    sources: list[str] = Field(default_factory=list, description="Citations")
    steps_used: int = Field(default=0, ge=0, description="Steps consumed")
    tokens_used: int = Field(default=0, ge=0, description="Tokens charged")

    @model_validator(mode="after")
    def validate_summary_for_ok_status(self) -> "WorkerResult":
        if self.status == "ok" and not self.summary.strip():
            raise ValueError(
                "Successful worker result (status='ok') must have a non-empty summary."
            )
        return self


# Dependency injection: Output validator
def validate_worker_result(raw: Any, expected_task_id: str) -> WorkerResult:
    """Validates raw output against WorkerResult contract and verifies task ID alignment."""
    if isinstance(raw, dict):
        result = WorkerResult.model_validate(raw)
    elif isinstance(raw, WorkerResult):
        result = raw
    else:
        raise TypeError(f"Worker returned invalid type: {type(raw).__name__}")

    if result.task_id != expected_task_id:
        raise ValueError(f"Task ID mismatch: expected '{expected_task_id}', got '{result.task_id}'")
    return result


async def run_worker_validated(
    task: SubTask,
    worker_fn: Callable[[SubTask], Coroutine[Any, Any, Any]],
    budget: Budget,
    timeout_s: float = 60.0,
    validator: Callable[[Any, str], WorkerResult] = validate_worker_result,
) -> WorkerResult:
    """Bounded worker wrapper with output validation: rejects malformed outputs without raising."""
    try:
        raw_output = await asyncio.wait_for(worker_fn(task), timeout=timeout_s)
        result = validator(raw_output, task.id)
    except TimeoutError:
        return WorkerResult(task_id=task.id, status="timeout")
    except Exception as exc:  # noqa: BLE001  # isolate worker exceptions & validation errors
        return WorkerResult(task_id=task.id, status="failed", summary=f"Rejected: {str(exc)[:200]}")

    if not await budget.charge(result.tokens_used):
        return WorkerResult(task_id=task.id, status="failed", summary="budget exceeded")
    return result

In [25]:
# --- Tests verifying rejection of malformed worker results ---

budget = Budget(max_tokens=1000)
task = SubTask(id="task_val", objective="Test output validation", output_format="text")


# 1. Reject invalid status (e.g. 'completed' instead of 'ok' | 'failed' | 'timeout')
async def worker_invalid_status(t: SubTask):
    return {"task_id": t.id, "status": "completed", "summary": "Finished"}


r1 = await run_worker_validated(task, worker_invalid_status, budget)
assert r1.status == "failed"
assert "Rejected" in r1.summary
print(f"✓ Rejected invalid status: {r1.summary[:75]}...")


# 2. Reject empty summary when status='ok'
async def worker_empty_summary(t: SubTask):
    return {"task_id": t.id, "status": "ok", "summary": "   "}


r2 = await run_worker_validated(task, worker_empty_summary, budget)
assert r2.status == "failed"
assert "non-empty summary" in r2.summary
print(f"✓ Rejected empty summary on ok: {r2.summary[:75]}...")


# 3. Reject negative token count
async def worker_negative_tokens(t: SubTask):
    return {"task_id": t.id, "status": "ok", "summary": "Done", "tokens_used": -20}


r3 = await run_worker_validated(task, worker_negative_tokens, budget)
assert r3.status == "failed"
print(f"✓ Rejected negative token count: {r3.summary[:75]}...")


# 4. Accept well-formed output
async def worker_valid(t: SubTask):
    return {"task_id": t.id, "status": "ok", "summary": "Valid finding", "tokens_used": 50}


r4 = await run_worker_validated(task, worker_valid, budget)
assert r4.status == "ok"
assert r4.summary == "Valid finding"
assert r4.tokens_used == 50
print("✓ Conforming output successfully validated and accepted!")

✓ Rejected invalid status: Rejected: 1 validation error for WorkerResult
status
  Input should be 'ok'...
✓ Rejected empty summary on ok: Rejected: 1 validation error for WorkerResult
  Value error, Successful wor...
✓ Rejected negative token count: Rejected: 1 validation error for WorkerResult
tokens_used
  Input should be...
✓ Conforming output successfully validated and accepted!


### Task 4


In [27]:
class RunMetrics(NamedTuple):
    pattern: str
    calls: int
    tokens: int
    latency: float
    response: str


BILLING_PROMPT = (
    "You are a Billing Specialist. Answer billing and payment questions concisely in 2 sentences."
)
REFUND_PROMPT = (
    "You are a Refund Specialist. Answer refund eligibility questions concisely in 2 sentences."
)


# Pattern 1: Handoffs (Sequential transfer of ownership to specialist)
async def run_handoff_triage(
    user_query: str,
    client: Client = llm_client,
    model: str = "gemini-3.5-flash-lite",
) -> RunMetrics:
    t0 = time.time()
    calls = 0
    tokens = 0

    # Step 1: Triage classifies and routes to specialist
    triage_prompt = (
        f"Route request to either 'billing' or 'refund'. Reply ONLY with the word.\n"
        f"Request: {user_query}"
    )
    resp1 = await client.aio.models.generate_content(model=model, contents=triage_prompt)
    calls += 1
    tokens += resp1.usage_metadata.total_token_count or 0
    route = resp1.text.strip().lower()

    # Step 2: Handoff - Chosen specialist directly answers the customer
    spec_sys = REFUND_PROMPT if "refund" in route else BILLING_PROMPT
    resp2 = await client.aio.models.generate_content(
        model=model, contents=f"{spec_sys}\nCustomer: {user_query}"
    )
    calls += 1
    tokens += resp2.usage_metadata.total_token_count or 0

    return RunMetrics("Handoffs", calls, tokens, round(time.time() - t0, 3), resp2.text.strip())


# Pattern 2: Subagents-as-Tools (Manager retains control and calls specialist as a tool)
async def run_subagents_as_tools_triage(
    user_query: str,
    client: Client = llm_client,
    model: str = "gemini-3.5-flash-lite",
) -> RunMetrics:
    t0 = time.time()
    calls = 0
    tokens = 0

    # Step 1: Manager decides which specialist tool to invoke
    mgr_prompt = (
        f"Available specialist tools: [billing_tool, refund_tool].\n"
        f"Which tool should resolve: '{user_query}'? Return ONLY tool name."
    )
    resp1 = await client.aio.models.generate_content(model=model, contents=mgr_prompt)
    calls += 1
    tokens += resp1.usage_metadata.total_token_count or 0
    tool = resp1.text.strip().lower()

    # Step 2: Specialist tool executes bounded lookup
    spec_sys = REFUND_PROMPT if "refund" in tool else BILLING_PROMPT
    resp2 = await client.aio.models.generate_content(
        model=model, contents=f"{spec_sys}\nTask lookup: {user_query}"
    )
    calls += 1
    tokens += resp2.usage_metadata.total_token_count or 0

    # Step 3: Manager synthesizes final reply to customer
    synth_prompt = (
        f"You are Support Manager. Synthesize customer reply in 2 sentences based on tool result.\n"
        f"Customer Query: {user_query}\n"
        f"Specialist Output: {resp2.text.strip()}"
    )
    resp3 = await client.aio.models.generate_content(model=model, contents=synth_prompt)
    calls += 1
    tokens += resp3.usage_metadata.total_token_count or 0

    return RunMetrics(
        "Subagents-as-Tools", calls, tokens, round(time.time() - t0, 3), resp3.text.strip()
    )

In [28]:
# Compare Handoffs vs Subagents-as-Tools on identical triage query
query = "I was charged twice ($49.99) for my subscription renewal this morning. Can I get a refund for the duplicate charge?"

print(f"Customer Query: {query}\n")

m_handoff = await run_handoff_triage(query)
m_tools = await run_subagents_as_tools_triage(query)

print("| Pattern | Calls | Tokens | Latency (s) |")
print("|---|---|---|---|")
print(f"| {m_handoff.pattern} | {m_handoff.calls} | {m_handoff.tokens} | {m_handoff.latency}s |")
print(f"| {m_tools.pattern} | {m_tools.calls} | {m_tools.tokens} | {m_tools.latency}s |")

print("\n--- Handoff Response ---")
print(m_handoff.response)

print("\n--- Subagents-as-Tools Response ---")
print(m_tools.response)

Customer Query: I was charged twice ($49.99) for my subscription renewal this morning. Can I get a refund for the duplicate charge?

| Pattern | Calls | Tokens | Latency (s) |
|---|---|---|---|
| Handoffs | 2 | 141 | 11.335s |
| Subagents-as-Tools | 3 | 299 | 27.35s |

--- Handoff Response ---
Yes, you are fully eligible for a refund for the duplicate $49.99 charge. I have processed the reversal, and the funds will appear in your account within 3 to 5 business days.

--- Subagents-as-Tools Response ---
You are fully eligible for a refund, and I have successfully processed the reversal of the duplicate $49.99 charge. The funds should appear on your original payment method within 3 to 5 business days.


### Task 5


In [29]:
def handoff_history_filter(
    history: list[dict],
    triage_summary: str,
    allowed_topics: tuple[str, ...] = ("login", "lock", "account", "security"),
) -> list[dict]:
    """Handoff history filter: strips tool calls and unrelated/sensitive messages, passing only relevant user context + summary."""
    clean_history = []
    for msg in history:
        # Drop internal tool calls, system notes, and assistant responses
        if msg.get("role") in ("tool", "system"):
            continue
        # Retain only user messages relevant to destination specialist topic
        content = msg.get("content", "")
        if msg.get("role") == "user" and any(kw in content.lower() for kw in allowed_topics):
            clean_history.append({"role": "user", "content": content})

    # Append the bounded 1-line triage summary
    clean_history.append({"role": "system", "content": f"Triage Summary: {triage_summary}"})
    return clean_history

In [30]:
# Raw conversation history containing sensitive billing info & internal tool execution
raw_history = [
    {"role": "user", "content": "My credit card 4111-2222-3333-4444 was charged $99 with CVV 888."},
    {"role": "tool", "content": "get_billing_data(token='sk_live_supersecret123')"},
    {"role": "assistant", "content": "Billing details confirmed."},
    {
        "role": "user",
        "content": "Now my account is locked and I cannot login due to suspicious activity.",
    },
]

# Apply filter for handoff to Account Security Specialist
filtered_history = handoff_history_filter(
    history=raw_history,
    triage_summary="Customer account is locked. Needs security verification.",
)

# 1. Deterministic assertions: verify sensitive data is completely stripped
assert "4111" not in str(filtered_history), "Credit card leaked!"
assert "888" not in str(filtered_history), "CVV leaked!"
assert "sk_live" not in str(filtered_history), "API Secret token leaked!"
print(
    "✓ Deterministic Check: Tool calls, credit card, and secret tokens completely removed from history."
)
print(f"Filtered context passed to specialist:\n{filtered_history}\n")

# 2. LLM probe: Verify Security Specialist cannot see or leak the hidden data
probe_prompt = (
    "You are an Account Security Specialist.\n"
    f"Provided Context:\n{filtered_history}\n\n"
    "Question: What credit card number, CVV, or secret token did the user provide?"
)
resp = await llm_client.aio.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=probe_prompt,
)
print("--- Security Specialist Response ---")
print(resp.text.strip())

✓ Deterministic Check: Tool calls, credit card, and secret tokens completely removed from history.
Filtered context passed to specialist:
[{'role': 'user', 'content': 'Now my account is locked and I cannot login due to suspicious activity.'}, {'role': 'system', 'content': 'Triage Summary: Customer account is locked. Needs security verification.'}]

--- Security Specialist Response ---
Based on the provided context, the user did not provide any credit card number, CVV, or secret token. Their message only stated that their account is locked due to suspicious activity.


### Task 6

In [33]:
@dataclass
class DelegationTracker:
    """Guards multi-agent delegation chains against recursion depth and routing loops."""

    max_depth: int = 3
    history: list[tuple[str, str, str]] = field(default_factory=list)

    def check(self, from_agent: str, to_agent: str, task: str) -> None:
        triple = (from_agent, to_agent, task)
        if len(self.history) >= self.max_depth:
            raise RecursionError(f"Delegation depth limit ({self.max_depth}) exceeded.")
        if triple in self.history:
            raise RuntimeError(f"Delegation loop detected on triple: {triple}")
        self.history.append(triple)


async def delegate(
    from_agent: str,
    to_agent: str,
    task: str,
    handler: Callable[[str, str, str, DelegationTracker], Coroutine[Any, Any, str]],
    tracker: DelegationTracker | None = None,
) -> str:
    """Delegates task with injected tracker enforcing depth and loop detection."""
    tracker = tracker or DelegationTracker()
    try:
        tracker.check(from_agent, to_agent, task)
        return await handler(from_agent, to_agent, task, tracker)
    except (RecursionError, RuntimeError) as err:
        return f"HALTED: {err}"

In [34]:
# --- Deliberate Triggers: Loop Detection & Depth Limits ---

# 1. Deliberately trigger loop detection: Agent A <-> Agent B bouncing the same task
async def ping_pong_handler(
    from_agent: str, to_agent: str, task: str, tracker: DelegationTracker
) -> str:
    next_recipient = "agent_a" if to_agent == "agent_b" else "agent_b"
    return await delegate(to_agent, next_recipient, task, ping_pong_handler, tracker)


loop_tracker = DelegationTracker(max_depth=10)
loop_result = await delegate(
    "agent_a", "agent_b", "review_contract", ping_pong_handler, loop_tracker
)

assert "Delegation loop detected" in loop_result
print("✓ Loop Detection Verified:")
print(f"  {loop_result}")
print(f"  Recorded history: {loop_tracker.history}\n")


# 2. Deliberately trigger depth limit: Chain A -> B -> C -> D exceeding max_depth=3
async def chain_handler(
    from_agent: str, to_agent: str, task: str, tracker: DelegationTracker
) -> str:
    next_agent = f"agent_{chr(ord(to_agent[-1]) + 1)}"  # agent_b -> agent_c -> agent_d
    return await delegate(to_agent, next_agent, f"subtask_{next_agent}", chain_handler, tracker)


depth_tracker = DelegationTracker(max_depth=3)
depth_result = await delegate("agent_a", "agent_b", "subtask_b", chain_handler, depth_tracker)

assert "Delegation depth limit (3) exceeded" in depth_result
print("✓ Delegation Depth Limit Verified:")
print(f"  {depth_result}")
print(f"  Hops executed before halting: {len(depth_tracker.history)}")

✓ Loop Detection Verified:
  HALTED: Delegation loop detected on triple: ('agent_a', 'agent_b', 'review_contract')
  Recorded history: [('agent_a', 'agent_b', 'review_contract'), ('agent_b', 'agent_a', 'review_contract')]

✓ Delegation Depth Limit Verified:
  HALTED: Delegation depth limit (3) exceeded.
  Hops executed before halting: 3


### Task 7

In [35]:
PRICE_PER_M_TOKENS = 0.15  # $0.15 per 1M tokens for gemini-3.1-flash-lite


async def safe_call(
    prompt: str,
    client: Client = llm_client,
    model: str = "gemini-3.1-flash-lite",
) -> tuple[str, int]:
    """Robust LLM call with quota backoff to respect rate limits."""
    for attempt in range(4):
        try:
            resp = await client.aio.models.generate_content(model=model, contents=prompt)
            return (resp.text or "").strip(), resp.usage_metadata.total_token_count or 0
        except Exception as exc:  # noqa: BLE001
            if "429" in str(exc) or "RESOURCE_EXHAUSTED" in str(exc):
                await asyncio.sleep(15 * (attempt + 1))
            else:
                await asyncio.sleep(2 * (attempt + 1))
    return "Call failed", 0


def score_quality(text: str) -> float:
    """Scores quality on a 1.0 - 5.0 scale evaluating completeness, cost analysis, and trade-offs."""
    score = 2.0
    lower = text.lower()
    if any(
        k in lower for k in ("architecture", "mechanism", "structure", "engine", "design", "data")
    ):
        score += 1.0
    if any(k in lower for k in ("cost", "pricing", "overhead", "resource", "budget", "expense")):
        score += 1.0
    if any(
        k in lower
        for k in ("trade-off", "vs", "versus", "advantage", "disadvantage", "pros", "cons")
    ):
        score += 1.0
    if len(text.split()) < 30:
        score = max(1.0, score - 1.0)
    return min(5.0, score)


async def single_agent_baseline(
    topic: str,
    client: Client = llm_client,
    model: str = "gemini-3.1-flash-lite",
) -> tuple[str, int, float, float]:
    """Single agent receives prompt and answers across all pillars in one pass."""
    t0 = time.time()
    prompt = (
        f"Analyze {topic} covering: 1. Architecture, 2. Cost impact, 3. Key trade-offs. Be concise."
    )
    text, tokens = await safe_call(prompt, client, model)
    cost = (tokens / 1_000_000) * PRICE_PER_M_TOKENS
    return text, tokens, cost, round(time.time() - t0, 2)


async def multi_agent_system(
    topic: str,
    client: Client = llm_client,
    model: str = "gemini-3.1-flash-lite",
) -> tuple[str, int, float, float]:
    """Multi-agent orchestrator: runs parallel specialist subtasks and synthesizes final answer."""
    t0 = time.time()
    p_arch = f"You are a Systems Architect. Detail architecture & mechanisms of: {topic}. 2 concise bullets."
    p_cost = f"You are an Operations Lead. Detail cost & resource trade-offs of: {topic}. 2 concise bullets."

    r_arch, r_cost = await asyncio.gather(
        safe_call(p_arch, client, model),
        safe_call(p_cost, client, model),
    )
    tokens_w = r_arch[1] + r_cost[1]

    synth_prompt = (
        f"Synthesize these specialist points into a final comparative report for {topic}:\n"
        f"Architecture:\n{r_arch[0]}\n\nCost/Ops:\n{r_cost[0]}"
    )
    synth_text, synth_tokens = await safe_call(synth_prompt, client, model)
    total_tokens = tokens_w + synth_tokens
    total_cost = (total_tokens / 1_000_000) * PRICE_PER_M_TOKENS
    return synth_text, total_tokens, total_cost, round(time.time() - t0, 2)

In [36]:
# 10 Comparative Architecture Topics
INPUTS = [
    "PostgreSQL vs MongoDB for high-throughput event logging",
    "Microservices vs Monolith architectures for early-stage startups",
    "REST APIs vs GraphQL for mobile client applications",
    "Redis vs Memcached for distributed cache clusters",
    "Docker containers vs WebAssembly for serverless edge compute",
    "Apache Kafka vs RabbitMQ for real-time messaging pipelines",
    "Single-Agent vs Multi-Agent architectures for code review",
    "RAG vs Model Fine-tuning for enterprise question answering",
    "Vector Search vs BM25 Keyword Search for legal document search",
    "Serverless Functions vs Dedicated Containers for background jobs",
]

print("Evaluating 10 inputs across Single-Agent Baseline vs Multi-Agent System...\n")

records = []
for idx, topic in enumerate(INPUTS, start=1):
    s_text, s_tokens, s_cost, s_lat = await single_agent_baseline(topic)
    m_text, m_tokens, m_cost, m_lat = await multi_agent_system(topic)
    s_score = score_quality(s_text)
    m_score = score_quality(m_text)
    records.append(
        {
            "id": idx,
            "topic": topic[:32] + "...",
            "s_score": s_score,
            "s_tokens": s_tokens,
            "s_cost": s_cost,
            "s_lat": s_lat,
            "m_score": m_score,
            "m_tokens": m_tokens,
            "m_cost": m_cost,
            "m_lat": m_lat,
        }
    )
    # Brief delay between items to respect API rate limits
    await asyncio.sleep(2.0)

# Display Comparison Table
print(
    "| # | Topic | Single Quality | Single Tokens | Single Cost | Multi Quality | Multi Tokens | Multi Cost |"
)
print("|---|---|---|---|---|---|---|---|")
for r in records:
    print(
        f"| {r['id']} | {r['topic']} | {r['s_score']:.1f}/5 | {r['s_tokens']} | ${r['s_cost']:.5f} | {r['m_score']:.1f}/5 | {r['m_tokens']} | ${r['m_cost']:.5f} |"
    )

avg_s_score = sum(r["s_score"] for r in records) / len(records)
avg_m_score = sum(r["m_score"] for r in records) / len(records)
avg_s_tokens = sum(r["s_tokens"] for r in records) / len(records)
avg_m_tokens = sum(r["m_tokens"] for r in records) / len(records)
avg_s_cost = sum(r["s_cost"] for r in records) / len(records)
avg_m_cost = sum(r["m_cost"] for r in records) / len(records)

print("\n=== Summary Averages ===")
print(
    f"Single-Agent: Avg Quality = {avg_s_score:.2f}/5 | Avg Tokens = {avg_s_tokens:.1f} | Avg Cost = ${avg_s_cost:.5f}"
)
print(
    f"Multi-Agent:  Avg Quality = {avg_m_score:.2f}/5 | Avg Tokens = {avg_m_tokens:.1f} | Avg Cost = ${avg_m_cost:.5f}"
)
print(
    f"Token Multiplier: {avg_m_tokens / max(avg_s_tokens, 1):.2f}x | Quality Delta: {avg_m_score - avg_s_score:+.2f}"
)

Evaluating 10 inputs across Single-Agent Baseline vs Multi-Agent System...

| # | Topic | Single Quality | Single Tokens | Single Cost | Multi Quality | Multi Tokens | Multi Cost |
|---|---|---|---|---|---|---|---|
| 1 | PostgreSQL vs MongoDB for high-t... | 5.0/5 | 670 | $0.00010 | 5.0/5 | 1472 | $0.00022 |
| 2 | Microservices vs Monolith archit... | 5.0/5 | 622 | $0.00009 | 5.0/5 | 1589 | $0.00024 |
| 3 | REST APIs vs GraphQL for mobile ... | 5.0/5 | 575 | $0.00009 | 5.0/5 | 1403 | $0.00021 |
| 4 | Redis vs Memcached for distribut... | 5.0/5 | 538 | $0.00008 | 5.0/5 | 1624 | $0.00024 |
| 5 | Docker containers vs WebAssembly... | 5.0/5 | 568 | $0.00009 | 5.0/5 | 1609 | $0.00024 |
| 6 | Apache Kafka vs RabbitMQ for rea... | 5.0/5 | 635 | $0.00010 | 5.0/5 | 1596 | $0.00024 |
| 7 | Single-Agent vs Multi-Agent arch... | 5.0/5 | 608 | $0.00009 | 5.0/5 | 1669 | $0.00025 |
| 8 | RAG vs Model Fine-tuning for ent... | 5.0/5 | 588 | $0.00009 | 5.0/5 | 1616 | $0.00024 |
| 9 | Vector Search vs BM

### Task 8

In [38]:
@dataclass
class TraceSpan:
    """Represents a single bounded execution span within a multi-agent run."""

    run_id: str
    span_id: str
    parent_id: str | None
    agent_name: str
    inputs: Any
    outputs: Any
    tokens: int
    latency: float


class TraceLogger:
    """Collects and serializes multi-agent execution spans into JSON lines."""

    def __init__(self) -> None:
        self.spans: list[TraceSpan] = []

    def record(
        self,
        run_id: str,
        agent_name: str,
        inputs: Any,
        outputs: Any,
        tokens: int,
        latency: float,
        parent_id: str | None = None,
    ) -> TraceSpan:
        span = TraceSpan(
            run_id=run_id,
            span_id=str(uuid.uuid4())[:8],
            parent_id=parent_id,
            agent_name=agent_name,
            inputs=inputs,
            outputs=outputs,
            tokens=tokens,
            latency=latency,
        )
        self.spans.append(span)
        return span

    def to_jsonl(self) -> str:
        """Serializes recorded spans to newline-delimited JSON."""
        return "\n".join(json.dumps(asdict(s)) for s in self.spans)


def reconstruct_run(jsonl_logs: str, run_id: str) -> list[dict[str, Any]]:
    """Reconstructs the hierarchical delegation tree of a multi-agent run from JSONL logs."""
    spans = [json.loads(line) for line in jsonl_logs.splitlines() if line.strip()]
    run_spans = [s for s in spans if s["run_id"] == run_id]

    by_parent: dict[str | None, list[dict[str, Any]]] = {}
    for s in run_spans:
        by_parent.setdefault(s["parent_id"], []).append(s)

    def print_tree(parent_id: str | None, indent: int = 0) -> None:
        for span in by_parent.get(parent_id, []):
            prefix = "  " * indent + ("└── " if indent > 0 else "")
            name = span["agent_name"]
            sid = span["span_id"]
            pid = span["parent_id"]
            tok = span["tokens"]
            lat = span["latency"]
            print(f"{prefix}[{name}] (span={sid}, parent={pid}, tokens={tok}, latency={lat}s)")
            print_tree(span["span_id"], indent + 1)

    print(f"=== Reconstructed Execution Tree for Run: {run_id} ===")
    print_tree(None)
    return run_spans

In [39]:
# --- Execute a Traced Multi-Agent Run & Reconstruct from JSONL Logs ---

tracer = TraceLogger()
test_run_id = f"run_{str(uuid.uuid4())[:8]}"

# 1. Root Lead Orchestrator span (parent=None)
lead_span = tracer.record(
    run_id=test_run_id,
    agent_name="lead_orchestrator",
    inputs={"goal": "Compare PostgreSQL vs MongoDB"},
    outputs="Delegating to specialized workers",
    tokens=0,
    latency=0.05,
    parent_id=None,
)

# 2. Child Specialist Spans (parent_id = lead_span.span_id)
w1_span = tracer.record(
    run_id=test_run_id,
    agent_name="systems_architect",
    inputs="Analyze PostgreSQL relational architecture",
    outputs="ACID compliance, relational schema, B-tree indexes",
    tokens=145,
    latency=1.20,
    parent_id=lead_span.span_id,
)

w2_span = tracer.record(
    run_id=test_run_id,
    agent_name="operations_lead",
    inputs="Analyze MongoDB operational cost overhead",
    outputs="Higher RAM requirements, horizontal sharding complexity",
    tokens=160,
    latency=1.15,
    parent_id=lead_span.span_id,
)

# 3. Final Synthesis Span (parent_id = lead_span.span_id)
synth_span = tracer.record(
    run_id=test_run_id,
    agent_name="report_synthesizer",
    inputs={"architect_findings": w1_span.outputs, "ops_findings": w2_span.outputs},
    outputs="Comprehensive comparative report completed.",
    tokens=110,
    latency=0.85,
    parent_id=lead_span.span_id,
)

# Export to JSON Lines log
jsonl_logs = tracer.to_jsonl()

print("--- Generated JSON Lines Log ---")
for line in jsonl_logs.splitlines():
    print(line)
print()

# Reconstruct the execution tree purely from the JSON Lines logs
reconstructed_spans = reconstruct_run(jsonl_logs, test_run_id)
assert len(reconstructed_spans) == 4
print(
    "\n✓ Verification successful: Entire multi-agent run reconstructed with parent-child linkage."
)

--- Generated JSON Lines Log ---
{"run_id": "run_88d6de53", "span_id": "808e6ff0", "parent_id": null, "agent_name": "lead_orchestrator", "inputs": {"goal": "Compare PostgreSQL vs MongoDB"}, "outputs": "Delegating to specialized workers", "tokens": 0, "latency": 0.05}
{"run_id": "run_88d6de53", "span_id": "67761d81", "parent_id": "808e6ff0", "agent_name": "systems_architect", "inputs": "Analyze PostgreSQL relational architecture", "outputs": "ACID compliance, relational schema, B-tree indexes", "tokens": 145, "latency": 1.2}
{"run_id": "run_88d6de53", "span_id": "370f9a9c", "parent_id": "808e6ff0", "agent_name": "operations_lead", "inputs": "Analyze MongoDB operational cost overhead", "outputs": "Higher RAM requirements, horizontal sharding complexity", "tokens": 160, "latency": 1.15}
{"run_id": "run_88d6de53", "span_id": "996459ac", "parent_id": "808e6ff0", "agent_name": "report_synthesizer", "inputs": {"architect_findings": "ACID compliance, relational schema, B-tree indexes", "ops_fi